# Zadanie 2.1: solvery, rozdzielczość i czas obliczeń

Porównujemy dwa schematy numeryczne dla modelu
$\partial_t u = D\,\partial_{xx} u + \rho u (1 - u/K) - \beta R(x) U(t) u$:

- **A, `rk4`**: różnice centralne w $x$ i jawny RK4. Krok czasu ogranicza dyfuzja ($\Delta t \le 0{,}4\,\Delta x^2/D$)
  i szerokość impulsu dawki ($\Delta t \le w/5$).
- **B, `cn`**: rozszczepienie Stranga. Dyfuzja schematem Cranka-Nicolsona (niejawnym, stabilnym dla każdego $\Delta t$),
  reakcja i dawka dokładnie w każdym węźle. Krok czasu wybieramy dowolnie; tu 0,02 i 0,1.

Scenariusz: wiązka przesunięta ($s = 6$, $r = 8$, $\beta = 1{,}5$), 10 frakcji co 0,4, $t \in [0, 10]$, obszar długości 80.
Odniesienie: solver B na siatce $\Delta x = 0{,}0125$ z krokiem $10^{-3}$.

In [ ]:
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, grid
from offtarget.observables import time_to_regrowth, tumor_mass
from offtarget.solvers import simulate

rng = seed_everything()

L = 80.0
params = Params(beta=1.5, s=6.0, r=8.0)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
t = np.round(np.linspace(0.0, 10.0, 101), 10)

## Rozwiązanie odniesienia

Sprawdzamy też, że solver A na gęstej siatce daje to samo co odniesienie, czyli że oba schematy zbiegają do jednego rozwiązania.

In [ ]:
DX_REF = 0.0125
x_ref, _ = grid(L, int(round(L / DX_REF)))
tic = time.perf_counter()
u_ref = simulate(params, schedule, x_ref, t, method="cn", dt_max=1e-3)
print(f"odniesienie: {len(x_ref)} węzłów, {time.perf_counter() - tic:.1f} s")
M_ref = tumor_mass(u_ref, x_ref)
t_regrowth_ref = time_to_regrowth(t, M_ref)

x_chk, _ = grid(L, int(round(L / 0.025)))
M_chk = tumor_mass(simulate(params, schedule, x_chk, t, method="rk4"), x_chk)
print(f"solver A, dx = 0,025, wobec odniesienia: największy błąd względny M = {np.max(np.abs(M_chk - M_ref) / M_ref):.1e}")
print(f"czas do odrostu w odniesieniu: {t_regrowth_ref}")

## Siatki i pomiar

Dla każdego $\Delta x$: największy błąd względny masy guza $M(t)$ na całym przedziale, największy błąd pola $u$ w chwili końcowej
i czas obliczeń (najlepszy z trzech powtórzeń, CPU). Siatki są zagnieżdżone w siatce odniesienia, więc pole porównujemy w tych samych węzłach.

In [ ]:
DX = [1.0, 0.5, 0.2, 0.1, 0.05]
VARIANTS = [
    ("A: RK4", "rk4", None),
    ("B: CN, dt = 0,02", "cn", 0.02),
    ("B: CN, dt = 0,1", "cn", 0.1),
]


def best_time(fn, repeats=3):
    best, out = np.inf, None
    for _ in range(repeats):
        tic = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - tic)
    return best, out


rows = []
for name, method, dt in VARIANTS:
    for dx in DX:
        n = int(round(L / dx))
        x, _ = grid(L, n)
        seconds, u = best_time(lambda: simulate(params, schedule, x, t, method=method, dt_max=dt))
        M = tumor_mass(u, x)
        stride = int(round(dx / DX_REF))
        rows.append({
            "wariant": name, "dx": dx, "węzły": n + 1,
            "błąd M (względny)": np.max(np.abs(M - M_ref) / M_ref),
            "błąd u(T)": np.max(np.abs(u[-1] - u_ref[-1, ::stride])),
            "czas do odrostu": time_to_regrowth(t, M),
            "czas [s]": seconds,
        })
table = pd.DataFrame(rows)
table.to_csv(TABLES / "zad2_01_rozdzielczosc.csv", index=False)
print(f"Python {platform.python_version()}, {platform.machine()}; czasy zależą od komputera")
table.style.format({"błąd M (względny)": "{:.1e}", "błąd u(T)": "{:.1e}", "czas [s]": "{:.3f}"})

## Wykresy: zbieżność i błąd a czas

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for name, g in table.groupby("wariant", sort=False):
    a1.loglog(g["dx"], g["błąd M (względny)"], "o-", label=name)
    a2.loglog(g["czas [s]"], g["błąd M (względny)"], "o-", label=name)
    for _, r in g.iterrows():
        a2.annotate(f"{r['dx']:g}", (r["czas [s]"], r["błąd M (względny)"]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
dx_line = np.array([0.05, 1.0])
a1.loglog(dx_line, 2e-3 * dx_line**2, "k--", lw=1, label="nachylenie 2")
a1.set(xlabel="Δx", ylabel="największy błąd względny M(t)", title="Zbieżność względem siatki")
a2.set(xlabel="czas obliczeń [s]", ylabel="największy błąd względny M(t)", title="Błąd a czas (etykiety: Δx)")
for a in (a1, a2):
    a.grid(True, which="both", alpha=0.3)
    a.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_01_blad_czas.png", dpi=150)

## Wnioski

Liczby z przebiegu 19.11.2026 na MacBooku (Apple Silicon); czasy na innym komputerze będą inne, stosunki podobne.

- **Oba solvery mają rząd 2 w $\Delta x$**: błąd masy maleje ok. 4 razy przy siatce dwa razy gęstszej
  (np. A: $7{,}3\cdot10^{-4}$ przy $\Delta x = 0{,}2$, $1{,}8\cdot10^{-4}$ przy 0,1, $4{,}3\cdot10^{-5}$ przy 0,05).
  Solver A na siatce 0,025 zgadza się z odniesieniem do $9\cdot10^{-6}$, więc oba schematy zbiegają do tego samego rozwiązania.
- **Przy tej samej dokładności solver B jest szybszy**: przy $\Delta x = 0{,}05$ ok. 0,05 s wobec 0,76 s dla RK4 (ok. 16 razy).
  Jawny RK4 płaci za warunek $\Delta t \le 0{,}4\,\Delta x^2/D$: zagęszczenie siatki dwa razy to cztery razy mniejszy krok.
- **Krok czasu w solverze B ustala próg błędu**: przy $\Delta t = 0{,}1$ błąd nie spada poniżej ok. $1{,}1\cdot10^{-3}$,
  ale obliczenia trwają kilka milisekund. Do szybkich przeglądów parametrów (SA, mapa niedopasowania) to wystarczy.
- **Czas do odrostu** (5,5 w odniesieniu) zgadza się co do kroku zapisu 0,1 od $\Delta x = 0{,}1$; na rzadszych siatkach front biegnie
  za szybko i guz odrasta o 0,1 wcześniej.